# Hyperparameter Optimization in Google Colab

This notebook runs hyperparameter optimization using Optuna on Google Colab, here we use T4 GPU.

## 1. Environment Setup

Mount Google Drive for persistent storage and clone the repository.

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Define results directory on Drive
import os
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/optuna_results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results directory: {DRIVE_RESULTS_DIR}")

In [ ]:
# Clone or update repository
REPO_PATH = '/content/Progetto_deep_learning'
BRANCH = 'develop'

if not os.path.exists(REPO_PATH):
    print(f"Cloning branch {BRANCH}...")
    !git clone -b {BRANCH} https://github.com/scorzaluca/Progetto_deep_learning.git
else:
    print(f"Repository exists, updating branch {BRANCH}...")
    !cd {REPO_PATH} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

# Install dependencies and configure Python path
%pip install optuna -q
import sys
sys.path.insert(0, REPO_PATH)

print("\nSetup complete.")

## Configuration Instructions

Before running the optimization, update the configuration files:

1. Click the **folder icon** in the left sidebar
2. Navigate to: `/content/Progetto_deep_learning/src/config/`
3. Double-click `tuning_config.py` to open
4. Modify parameters and **save** (Ctrl+S)
5. **Re-run the cell below** to reload modules
6. Continue with subsequent cells

In [ ]:
# Re-run this cell after any configuration file modifications

import importlib
import src.config.tuning_config as tc
import src.config.training_config as trc
import src.config.model_config as mc
import src.config as cfg
import src.DataLoading as dl

# Reload all config modules
importlib.reload(tc)
importlib.reload(trc)
importlib.reload(mc)
importlib.reload(cfg)
importlib.reload(dl)

# Import updated configuration
from src.config import (
    SEED,
    TARGET_COL,
    MODEL_NAME,
    N_TRIALS,
    N_FOLDS,
    TUNING_EPOCHS,
    PATIENCE,
    STUDY_NAME,
    NEW_STUDY,
    NAIVE_MAE_FINAL_FOLD,
    EPOCHS,
)

# Verify configuration and device
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Model: {MODEL_NAME}")
print(f"Study: {STUDY_NAME} ({'NEW' if NEW_STUDY else 'RESUME'})")
print(f"Trials: {N_TRIALS}, Folds: {N_FOLDS}")
print(f"Epochs: {TUNING_EPOCHS}, Patience: {PATIENCE}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Data

Load the preprocessed dataset and create cross-validation folds.

In [ ]:
from src.Utils import set_seed, load_data_and_folds

DATA_PATH = f"{REPO_PATH}/data/processed/preprocessed_ds.csv"
STORAGE_PATH = f"{DRIVE_RESULTS_DIR}/optuna_studies.db"

set_seed(SEED)
df, folds = load_data_and_folds(data_path=DATA_PATH)

print(f"\nDataset shape: {df.shape}")
print(f"Number of folds: {len(folds)}")
print(f"Database path: {STORAGE_PATH}")

## 3. Optuna Optimization

Run hyperparameter optimization using the configured search space.

In [ ]:
from src.Tuning import OptunaOptimizer

optimizer = OptunaOptimizer(
    model_name=MODEL_NAME,
    folds=folds,
    device=DEVICE,
    config={
        "n_trials": N_TRIALS,
        "n_folds": N_FOLDS,
        "epochs": TUNING_EPOCHS,
        "patience": PATIENCE,
    },
    storage_path=STORAGE_PATH,
    verbose=False,
)

result = optimizer.optimize(
    study_name=STUDY_NAME,
    n_trials=N_TRIALS,
    new_study=NEW_STUDY,
)

print(f"\nBest MASE: {result['best_mase']:.4f}")
print(f"Best params: {result['best_params']}")

## 4. Visualization

Plot the optimization history showing trial progression.

In [ ]:
import optuna
import matplotlib.pyplot as plt

study = optuna.load_study(study_name=STUDY_NAME, storage=f"sqlite:///{STORAGE_PATH}")

values = [t.value for t in study.trials if t.value is not None]
best_values = [min(values[:i+1]) for i in range(len(values))]

plt.figure(figsize=(10, 4))
plt.plot(values, 'o-', alpha=0.6, label='Trial MASE')
plt.plot(best_values, 'r-', linewidth=2, label='Best MASE')
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Trial')
plt.ylabel('MASE')
plt.title(f'{MODEL_NAME.upper()} - Optimization History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_plot.png", dpi=150)
plt.show()

## 5. Final Model Training

Train the final model using the best hyperparameters on the complete training set.

In [ ]:
from src.Training.engine import create_model, fit_model
from src.DataLoading import create_final_train_val_loaders
from src.config import LOOKBACK

best_params = result["best_params"]
train_loader, val_loader, scaler = create_final_train_val_loaders(df, TARGET_COL, LOOKBACK)

model = create_model(MODEL_NAME, best_params)
model.to(DEVICE)

model, history, best_epoch = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=best_params.get("lr", 0.001),
    device=DEVICE,
    patience=10,
    optimizer_kwargs={"weight_decay": best_params.get("weight_decay", 0.001)},
    grad_clip_norm=best_params.get("grad_clip_norm", 1.0),
    verbose=True,
    baseline_mae=NAIVE_MAE_FINAL_FOLD,
)

print(f"\nFinal best MASE: {min(history['val_mase']):.4f}")

## 6. Save Results

Save the trained model weights, hyperparameters, and training history to Google Drive.

In [ ]:
import json

# Save model weights
torch.save(model.state_dict(), f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_model.pth")

# Save hyperparameters
with open(f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

# Save training history
with open(f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"Results saved to: {DRIVE_RESULTS_DIR}/")
!ls -la {DRIVE_RESULTS_DIR}/

## 7. List Database Studies

Display all Optuna studies stored in the database.

In [ ]:
import optuna
STORAGE_PATH="/content/drive/MyDrive/optuna_results/optuna_studies.db"

# List all available studies
all_studies = optuna.study.get_all_study_names(storage=f"sqlite:///{STORAGE_PATH}")
print("Available studies:")
for name in all_studies:
    print(f"  - {name}")